# Analyse der Lebenserwartung

Zwei Datensätze zur Untersuchung der globalen Lebenserwartung:
- **Gapminder** (1952–2007): country, year, population, continent, lifeExp, gdpPercap
- **WHO-Lebenserwartung** (2000–2015): 193 Länder, 22 Indikatoren (Mortalität, BMI, BIP, Schulbildung usw.)

Diese Arbeitsmappe demonstriert den CSV-Import und die Datenanalyse sowohl in **Python** als auch in **R**.

## 1. Einrichtung: Pakete installieren & Datensätze herunterladen

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('pandas + plotly installiert')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"Bereits vorhanden: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"Heruntergeladen: {name}: {lines} Zeilen")

## 2. Gapminder: Exploration mit Python

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"Dimension (Shape): {gap.shape}")
print(f"Kontinente: {sorted(gap['continent'].unique())}")
print(f"Jahresbereich: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# Lebenserwartung im Zeitverlauf nach Kontinent
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='Lebenserwartung nach Kontinent (1952–2007)',
              labels={'lifeExp': 'Lebenserwartung (Jahre)', 'year': 'Jahr'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# BIP vs. Lebenserwartung (2007), Blasengröße = Bevölkerung
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='BIP vs. Lebenserwartung (2007)',
                 labels={'gdpPercap': 'BIP pro Kopf (log)', 'lifeExp': 'Lebenserwartung'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: Exploration mit R

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# Verteilung der Lebenserwartung nach Kontinent (Boxplot)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "Lebenserwartung nach Kontinent",
        xlab = "Kontinent", ylab = "Lebenserwartung (Jahre)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# Top 10 Länder nach Anstieg der Lebenserwartung (1952 vs. 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "Top 10: Zuwachs an Lebenserwartung (1952–2007)",
        xlab = "Gewonnene Jahre",
        col = "#00CC96", border = NA)

## 4. WHO-Lebenserwartung: Exploration mit Python

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"Dimension (Shape): {who.shape}")
print(f"Spalten: {list(who.columns)}")
print(f"\nFehlende Werte (Top 5):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# Entwicklungs- vs. Industrieländer: Vorab gruppierte Verteilungen der Lebenserwartung
# Explizite Balkenkoordinaten rendern konsistent über die Browser-Plotly-Brücke.
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='Lebenserwartung: Entwicklungs- vs. Industrieländer',
             labels={'Life expectancy': 'Lebenserwartung (Jahre)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# Schulbildung vs. Lebenserwartung
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='Schulbildung vs. Lebenserwartung (2014)',
                 labels={'Life expectancy': 'Lebenserwartung (Jahre)',
                         'Schooling': 'Schuljahre'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. WHO-Lebenserwartung: Exploration mit R

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nLänder:", length(unique(who$Country)))
cat("\nJahresbereich:", range(who$Year))

In [ ]:
# Korrelation: Erwachsenenmortalität vs. Lebenserwartung
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "Erwachsenenmortalität vs. Lebenserwartung",
     xlab = "Erwachsenenmortalität (pro 1000)",
     ylab = "Lebenserwartung (Jahre)")
legend("topright", legend = c("Industrieländer", "Entwicklungsländer"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# Einfaches lineares Modell: Was prognostiziert die Lebenserwartung?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## Wichtigste Erkenntnisse

- Die Lebenserwartung ist weltweit gestiegen, es bestehen jedoch weiterhin große Unterschiede zwischen den Kontinenten
- BIP und Schulbildung sind starke positive Prädiktoren für die Lebenserwartung
- Die Erwachsenenmortalität ist der stärkste negative Prädiktor
- Entwicklungsländer weisen eine wesentlich größere Streuung bei den Ergebnissen auf